# ScamLens Phase 2 - baselines & honest evaluation

This notebook reads the files produced by the pipeline (no training here):

* `results/metrics.json` - written by `python -m src.evaluate`
* `results/RESULTS.md`  - the same numbers as tables (plus the appended audit)

**Remember the one rule: synthetic results are not performance claims.
Headline numbers come only from `test_real`.**

In [ ]:
# load the metrics written by src/evaluate.py
import json
from pathlib import Path

results = json.loads(Path("../results/metrics.json").read_text())
print("test sets:", list(results))
for split, models in results.items():
    print(f"  {split:10s} -> {list(models)}")

## 1. The headline table per test set

macro-F1, scam recall, false-positive rate on the HARD negatives
(legit messages full of bank words), and the multi-label tactic scores.

In [ ]:
import pandas as pd

rows = []
for split, models in results.items():
    for name, r in models.items():
        lo, hi = r["macro_f1_ci95"]
        rows.append({
            "test set": split,
            "model": name,
            "macro-F1": round(r["macro_f1"], 3),
            "CI low": round(lo, 3) if lo == lo else None,
            "CI high": round(hi, 3) if hi == hi else None,
            "scam recall": round(r["scam_recall"], 3),
            "FPR(hard neg)": round(r["fpr_on_hard_negatives"], 3),
            "tactics micro-F1": round(r["tactics"]["micro_f1"], 3),
            "tactics macro-F1": round(r["tactics"]["macro_f1"], 3),
        })
df = pd.DataFrame(rows)
df

## 2. Confusion matrices (verdict_lr)

Rows = gold, columns = predicted. A healthy matrix is bright on its diagonal;
bright off-diagonal cells show exactly which classes get confused.

In [ ]:
labels = ["safe", "scam", "suspicious"]
for split in results:
    cm = results[split]["verdict_lr"]["confusion_matrix"]
    print(f"--- {split} (rows=gold, cols=pred: {labels}) ---")
    print(pd.DataFrame(cm, index=labels, columns=labels).to_string(), "\n")

## 3. The gap check

macro-F1 on `test_a` (seen families) vs `test_b` (unseen families) vs
`test_real` (real messages). A big drop test_a -> test_b means memorisation;
a big drop test_b -> test_real means the model learned the GENERATOR.

In [ ]:
def macro(split, model="verdict_lr"):
    return results.get(split, {}).get(model, {}).get("macro_f1")

a, b, r = macro("test_a"), macro("test_b"), macro("test_real")
print(f"test_a   (seen families, synthetic): {a:.3f}" if a else "test_a: missing")
print(f"test_b   (unseen families, synthetic): {b:.3f}" if b else "test_b: missing")
print(f"test_real (REAL messages):            {r:.3f}" if r else "test_real: missing")
if a and b:
    print(f"\nGap test_a - test_b     = {a - b:+.3f}   (memorisation gap)")
if b and r:
    print(f"Gap test_b - test_real  = {b - r:+.3f}   (generator gap)")

## 4. Shortcut audit

The audit trains one tiny model per SHALLOW feature (length, `<URL>`, digits,
devanagari, bank word, exclamation count, caps ratio). Any single feature
above 0.60 macro-F1 means the generator leaks through it.

In [ ]:
from src.audit import extract_features
from src.baselines import load_models, predict_verdicts
from src.io_utils import load_messages

df_msg = load_messages()
test_a = df_msg[df_msg["split"] == "test_a"].reset_index(drop=True)
train = df_msg[(df_msg["split"] == "train") & (df_msg["source"] != "public_weak")]

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
import numpy as np

feats_train = pd.DataFrame([extract_features(t) for t in train["text"]])
feats_test = pd.DataFrame([extract_features(t) for t in test_a["text"]])
audit_rows = []
for col in feats_train.columns:
    clf = LogisticRegression(max_iter=1000, random_state=42).fit(
        feats_train[[col]], train["verdict"])
    pred = clf.predict(feats_test[[col]])
    score = f1_score(test_a["verdict"], pred,
                     labels=["safe", "scam", "suspicious"],
                     average="macro", zero_division=0)
    audit_rows.append({"feature": col, "macro-F1": round(score, 3),
                       "FLAGGED": score >= 0.60})
pd.DataFrame(audit_rows).sort_values("macro-F1", ascending=False)

## 5. Full audit report

Top model features for the scam class + the most confident wrong
predictions on `test_b` are appended to `results/RESULTS.md` by
`python -m src.audit`. Read them below.

In [ ]:
report = Path("../results/RESULTS.md").read_text(encoding="utf-8")
audit_part = report.split("# ScamLens - shortcut audit")[-1]
print("# ScamLens - shortcut audit" + audit_part[:4000])

## What I notice

1. **The honest number is low, and that is the point.** `verdict_lr` reaches
   macro-F1 **0.572** on `test_a` but only **0.316** on `test_real`
   (95% CI 0.313-0.318). Two thirds of the synthetic score was the generator,
   not the language of scams.
2. **It memorised the hard negatives instead of learning them.** On the
   synthetic hard negatives its false-positive rate is **0.000** - on the 405
   REAL safe messages in `test_real` it is **0.728**: it labels 295 of them
   `scam`. Synthetic hard negatives are near-copies of its training rows;
   real bank alerts are not.
3. **`suspicious` is effectively dead.** Across val/test_a/test_b recall for
   that class is ~0 (on `test_a`, all 87 suspicious rows are predicted
   `scam`), so the 3-way task collapses into a 2-way one and every macro-F1
   here is capped near 0.6 by construction.
4. **The gap check is unambiguous**: `test_a` 0.572 -> `test_b` 0.523
   (-0.049, small memorisation) -> `test_real` 0.316 (-0.207, the generator
   gap). The drop appears exactly where it should if the model learned my
   templates.
5. **The shortcut audit is clean after the length fix.** `msg_length` scored
   **0.643** macro-F1 on its own (a single feature!) before I added short
   one-line scams and long legit bank/promo messages; now it is **0.224** and
   the strongest shallow feature is `has_digits` at **0.360**.
6. **Tactic tagging cannot be judged yet.** Micro-F1 is 0.000-0.124 because
   the real test set (IMC25) only carries 9 of the 13 tags and almost no
   OTP/job/investment/loan examples - 4 tags have ~0 real support.
7. **The generator repeats itself**: 67% of its wrong test_b predictions are
   exact duplicate sentences, and the suspicious pool has 1,548 rows but only
   839 distinct texts. More variety in `generate_suspicious` is the next fix.